# 🚆 Drishti-Kavach: Enhanced Google Colab Cloud GPU Training

This notebook provides an automated cloud GPU training pipeline for **YOLO11-seg (RailDrishti)** on Google Colab (A100 / L4 / T4).

### ✨ Key Features:
1. **Persistent Cloud Checkpointing:** Every epoch checkpoint (`last.pt`) is automatically backed up to Google Drive (`MyDrive/drishti_checkpoints/`).
2. **Pause / Resume Capability:** If your session disconnects or you pause training, pick up right from the last saved epoch with 0% data loss.
3. **Minimal 3-Line Per-Epoch Metrics:** Color-coded accuracy evaluation (Track Segm mAP, Obstacle Detection mAP, and Overall mAP).
4. **Automatic Model Export:** Best model (`best.pt`) is automatically saved to `MyDrive/RailDrishti.pt`.

### ⚡ Step 1: Check GPU Acceleration
Set runtime to GPU: **Runtime -> Change runtime type -> T4 GPU / A100**.

In [ ]:
!nvidia-smi
import torch
print(f'PyTorch CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU Device: {torch.cuda.get_device_name(0)}')

### 📦 Step 2: Install Dependencies

In [ ]:
!pip install -q ultralytics opencv-python-headless shapely pyyaml

### 📂 Step 3: Mount Google Drive & Extract Dataset to Fast NVMe SSD
Make sure you have uploaded `raildrishti_colab.zip` to your Google Drive root folder (`MyDrive`).

In [ ]:
import os, shutil, zipfile
from pathlib import Path
from google.colab import drive

# 1. Mount Drive
drive.mount('/content/drive', force_remount=False)
drive_root = Path('/content/drive/MyDrive')
zip_path = drive_root / 'raildrishti_colab.zip'
target_dataset = Path('/content/dataset_rail-drishti')

# 2. Extract
if not (target_dataset / 'images' / 'train').exists():
    print(f'[*] Extracting {zip_path} to local NVMe...')
    with zipfile.ZipFile(str(zip_path), 'r') as zip_ref:
        zip_ref.extractall('/content')
    print('  ✓ Dataset extracted successfully!')
else:
    print('  ✓ Dataset already extracted and ready.')

# 3. Create Colab Dataset YAML
yaml_path = Path('/content/raildrishti_colab.yaml')
yaml_content = """path: /content/dataset_rail-drishti
train: images/train
val: images/val
names:
  0: Rail_Track_Bed
  1: Rail_Lines
  2: Person
  3: Car
  4: Truck
  5: Branch
  6: IronRod
  7: Barrel
  8: Boulder
  9: Jerrycan
"""
with open(yaml_path, 'w') as f:
    f.write(yaml_content)
print(f'  ✓ Dataset YAML configured at {yaml_path}')

### 📊 Step 4: Setup Accuracy Monitor & Drive Auto-Backup Callback

In [ ]:
def get_metric_color(val: float) -> str:
    if val >= 90.0: return '\033[1;92m'
    elif val >= 80.0: return '\033[92m'
    elif val >= 65.0: return '\033[93m'
    elif val >= 45.0: return '\033[95m'
    else: return '\033[91m'

class ColabAccuracyMonitor:
    TARGET_BOX_MAP = 85.0
    TARGET_SEG_MAP = 90.0
    TARGET_OVERALL_MAP = 85.0

    def __init__(self, drive_backup_dir: Path, drive_best_path: Path):
        self.best_map = 0.0
        self.best_epoch = 0
        self.drive_backup_dir = drive_backup_dir
        self.drive_best_path = drive_best_path
        self.drive_backup_dir.mkdir(parents=True, exist_ok=True)

    def on_fit_epoch_end(self, trainer):
        epoch = trainer.epoch + 1
        total_epochs = trainer.epochs
        metrics = getattr(trainer, 'metrics', {}) or {}
        box_map50 = (metrics.get('metrics/mAP50(B)', 0.0) or 0.0) * 100.0
        seg_map50 = (metrics.get('metrics/mAP50(M)', 0.0) or 0.0) * 100.0
        overall_map50 = (box_map50 + seg_map50) / 2.0 if (box_map50 > 0 and seg_map50 > 0) else (box_map50 or seg_map50 or 0.0)
        loss_val = float(trainer.tloss.mean()) if hasattr(trainer, 'tloss') and trainer.tloss is not None else None
        
        is_new_best = False
        if overall_map50 > self.best_map and overall_map50 > 1.0:
            self.best_map = overall_map50
            self.best_epoch = epoch
            is_new_best = True
            if hasattr(trainer, 'best') and os.path.exists(str(trainer.best)):
                try: shutil.copy(str(trainer.best), str(self.drive_best_path))
                except Exception: pass

        if hasattr(trainer, 'last') and os.path.exists(str(trainer.last)):
            try: shutil.copy(str(trainer.last), str(self.drive_backup_dir / 'last.pt'))
            except Exception: pass

        c_overall = get_metric_color(overall_map50)
        c_seg = get_metric_color(seg_map50)
        c_box = get_metric_color(box_map50)
        c_best = get_metric_color(self.best_map)
        rst = '\033[0m'
        loss_str = f'{loss_val:.4f}' if loss_val is not None else 'N/A'
        best_msg = f' (★ Backed up to Drive: {self.drive_best_path.name})' if is_new_best else ''

        print(f'\nEpoch [{epoch:02d}/{total_epochs:02d}] -> Current Accuracy: {c_overall}{overall_map50:.1f}%{rst} (Expected: ≥{self.TARGET_OVERALL_MAP:.0f}%) | Loss: {loss_str}')
        print(f'Metrics: Track Segm mAP: {c_seg}{seg_map50:.1f}%{rst} (Expected: ≥{self.TARGET_SEG_MAP:.0f}%) | Obstacle Box mAP: {c_box}{box_map50:.1f}%{rst} (Expected: ≥{self.TARGET_BOX_MAP:.0f}%)')
        print(f'Best Accuracy: {c_best}{self.best_map:.1f}%{rst} at Epoch {self.best_epoch}{best_msg}\n')

### 🚀 Step 5: Run Training (Fresh or Resume)
Set `RESUME_TRAINING = False` to start fresh, or `RESUME_TRAINING = True` to resume from the last checkpoint saved on Google Drive!

In [ ]:
from ultralytics import YOLO

# ── CONFIGURATION ──────────────────────────────────────────────────────────
RESUME_TRAINING = False       # Set to True to continue from last checkpoint
EPOCHS = 60
BATCH_SIZE = 32
IMGSZ = 640                  # 640 for fast training, or 1024 for max resolution
BASE_MODEL = 'yolo11s-seg.pt'
# ──────────────────────────────────────────────────────────────────────────

drive_ckpt_dir = Path('/content/drive/MyDrive/drishti_checkpoints')
drive_last_ckpt = drive_ckpt_dir / 'last.pt'
drive_best_model = Path('/content/drive/MyDrive/RailDrishti.pt')

monitor = ColabAccuracyMonitor(drive_backup_dir=drive_ckpt_dir, drive_best_path=drive_best_model)

if RESUME_TRAINING and drive_last_ckpt.exists():
    print(f'[*] Resuming training from Google Drive checkpoint: {drive_last_ckpt}...')
    local_ckpt = Path('/content/last.pt')
    shutil.copy(str(drive_last_ckpt), str(local_ckpt))
    model = YOLO(str(local_ckpt))
    model.add_callback('on_fit_epoch_end', monitor.on_fit_epoch_end)
    model.train(resume=True)
else:
    if RESUME_TRAINING:
        print(f'[!] No checkpoint found at {drive_last_ckpt}. Starting fresh instead...')
    print(f'[*] Starting fresh training with base weights: {BASE_MODEL}...')
    model = YOLO(BASE_MODEL)
    model.add_callback('on_fit_epoch_end', monitor.on_fit_epoch_end)
    model.train(
        data='/content/raildrishti_colab.yaml',
        epochs=EPOCHS,
        batch=BATCH_SIZE,
        imgsz=IMGSZ,
        device=0,
        workers=8,
        project='/content/runs',
        name='raildrishti_colab',
        exist_ok=True,
        amp=True,
        optimizer='AdamW',
        lr0=0.002,
        cos_lr=True,
        weight_decay=0.0005,
        warmup_epochs=3.0,
        patience=15,
        mosaic=1.0,
        mixup=0.1,
        fliplr=0.5,
        hsv_h=0.015,
        hsv_s=0.6,
        hsv_v=0.4
    )